# W07B Colab B ·이웃·평균보정·평가·마지막 앱

추천 시스템 ·동덕여자대학교 ·유원상 교수 ·2026년 10월 15일

CPU에서 위부터 실행하세요. 두 노트북은 각각 독립 실행됩니다. GPU·유료 API·필수 파일 업로드가 없습니다. **파일 → Drive에 사본 저장**으로 자신의 실습본을 만드세요. 설치 후 런타임 재시작 안내가 나오면 재시작하고 첫 셀부터 실행합니다.

문제 코드는 Markdown에 제시하고 자신의 답은 별도 셀에 씁니다. 빈칸이나 일부러 잘못된 코드를 자동으로 실행하지 않습니다. 따라서 전체 실행 성공은 시연 기능의 확인이며, 자신의 연습 문제 정답을 모두 풀었다는 뜻은 아닙니다. 설명용 데이터는 연습 문제 데이터와 별개입니다.


## 가져오는 코드의 정의를 바로 확인하기

아래 실행 셀은 설치한 고정 버전의 GitHub 파일과 **정의 시작 행**으로 연결합니다. 클래스와 메소드를 각각 확인할 수 있습니다. `help(함수)`로 입력·반환·대체 규칙을 읽어도 됩니다. 이 API를 외우는 것은 시험 문제의 요구가 아닙니다.

| 함수/클래스 | 입력 → 출력과 의미 |
|---|---|
| toy_ratings | 인자 없음 → 새 합성 관측18행. 없는 행은 미관측 |
| binary_jaccard | 두 행동 집합 → 교집합/합집합, 둘 다 비면 근거 없음 |
| UserCF.fit | 학습 관측 → 평균·마스크·유사도를 가진 모델 |
| EvidenceCF | 축/k/ 최소 공통/ 최소 이웃/ 보정/ 범위 제한 → 설정 객체 |
| EvidenceCF.fit | train 관측 → 같은 모델 반환; SGD가 아님 |
| EvidenceCF.predict_details | 사용자·영화 ID쌍 → 예측/원예측/근거/이웃수 |
| EvidenceCF.explain | 사용자ID,영화ID → 이웃/비중/기여도 표; 대체면 빈 표 |
| rating_report / evaluate_cf | 학습된 모델, 미사용 관측 / 추가 카탈로그 → 오차·순위 지표 |
| cf_review_data | 없음 → train15/validation3/ 카탈로그 5편 |
| cf_review_experiment | 설정 → 한 예측의 근거와 동일 검증 지표 dict |
| cf_review_view | 설정, 선택 설명 함수 → 앱 출력 6개; launch하지 않음 |
| build_cf_review_app | 선택 설명 함수 → Gradio 앱; launch는 마지막 셀 |
| definition_link | 함수·클래스·메소드, 고정ref → GitHub 정의줄 URL |

fit은 데이터에서 통계를 만드는 단계입니다. 예측 대상의 실제 정답은 전달하지 않습니다. 기본 코사인은 전체 좌표의 0 채움 표현을 사용하고, Pearson의 공통 좌표 비교와 구별합니다. `axis='item',centered=True`는 이 수업에서 지원하지 않아 명시적으로 거부합니다.


In [ ]:
import os,sys,subprocess,math
os.environ['GRADIO_ANALYTICS_ENABLED']='False'
IN_COLAB='google.colab' in sys.modules
COURSE_VERSION='2026-fall-w07b'
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',
        f'luna-recommender-course[apps] @ git+https://github.com/lunalab-ai/recommender.git@{COURSE_VERSION}',
        'gradio==6.26.0'],check=True)
import pandas as pd
from IPython.display import display,Markdown
from luna_recsys.collaborative import toy_ratings,UserCF,binary_jaccard
from luna_recsys.cf_synthesis import EvidenceCF,evaluate_cf,rating_report
from luna_recsys.cf_review import cf_review_data,cf_review_experiment,cf_review_view,build_cf_review_app
from luna_recsys.model_comparison import definition_link
print('Python',sys.version.split()[0], '수업 버전',COURSE_VERSION)


In [ ]:
apis=[toy_ratings,binary_jaccard,UserCF,UserCF.fit,EvidenceCF,EvidenceCF.fit,
 EvidenceCF.predict_details,EvidenceCF.explain,rating_report,evaluate_cf,
 cf_review_data,cf_review_experiment,cf_review_view,build_cf_review_app,definition_link]
display(Markdown('\n'.join(f'- [{f.__qualname__}]({definition_link(f,COURSE_VERSION)})' for f in apis)))


![사용자·아이템 CF가 참조하는 평점 칸](../../course/notion/assets/w07b/08-axes.png)

그림을 먼저 읽고 아래 코드의 행·열·분모와 연결하세요. 수업용 독자 제작 합성 예제입니다.


## 1. 학습 표와 검증 표를 먼저 분리

작은 관측 18건 중 (u2,B),(u3,A),(u4,D)를 validation으로 숨겼습니다. train15건으로만 평균·유사도를 계산합니다. 설정을 여러 번 바꾸는 실험이므로 **test 성능**이라고 부르지 않습니다. **확인 B1:** 왜 검증 정답을 원표에 합쳐 fit하면 안 될까요?


In [ ]:
data=cf_review_data()
display(data['train']);display(data['validation'])
print('train/validation sizes:',len(data['train']),len(data['validation']))


## 2. 평균보정한 조건만 바꾸기

같은 train과 같은 k에서 원평점 방식과 평균보정 방식을 비교합니다. 선택 이웃이 같더라도 value가 원평점인지 편차인지 읽으세요. **확인 B2:** 보정 방식의 contribution 합에 무엇을 더 해야 원예측이 될까요?


In [ ]:
for centered in [False,True]:
    result=cf_review_experiment(centered=centered)
    print('centered=',centered)
    display(result['details']);display(result['contributions'])


## 3. 최소 근거와 비교 축

설정을 한 번에 하나만 바꾸어 차이를 설명합니다. 첫 조건은 기본 user, 둘째는 최소 이웃 4, 셋째는 item입니다. 같은 검증 3건의 오차와 대체 비율을 함께 봅니다. 순위 지표의 대상은 관련 영화가 있는 u4 한 명뿐이라는 한계를 읽으세요.


In [ ]:
for settings in [dict(),dict(min_neighbors=4),dict(axis='item')]:
    r=cf_review_experiment(**settings)
    print('settings:',r['settings'])
    display(r['details']);display(r['contributions'])
    display(r['validation_predictions'])
    print(r['metrics'])


## 4. 기 초 Python 연습 문제

문제 번호는 학생 문제 편과 동일합니다. 각 문제의 입력은 독립적입니다. 코드가 실행되는지뿐 아니라, 주어진 관측·유사도·분모·선택 조건을 지켰는지 설명하세요.


### Q51 ·공통 수 필터 빈칸

rows 의각튜플은(ID, 유사도, 공통평가 수)이며 목표 평점은 모두 있다. 공통 수 2 이상, 양수 유사도만 남기도록 빈칸을 채우라.

```python
rows=[('A',0.9,1),('B',0.7,2),('C',0.4,3)]
eligible=[]
for uid,sim,common in rows:
    if sim>0 and ___:
        eligible.append(uid)
print(eligible)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q51=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q52 ·먼저 자른 Top-k 오류

rows 는 유사도 내림차순이고 각 튜플은(ID, 유사도, 목표 평점)이다. 유효 평가자를 먼저 고르고 k =2를 적용하도록 수정하라. [:k]는 앞 k 개를 뜻한다.

```python
rows=[('A',0.9,None),('B',0.8,5),('C',0.7,1)]
k=2
chosen=[r for r in rows[:k] if r[2] is not None]
print([r[0] for r in chosen])
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q52=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q53 ·최소 이웃 수 분기 추적

available은 k 적용 후 확보한 유효 이웃 수이다. 실행한 분기,used,pred,basis를 쓰라.

```python
available=2
min_neighbors=3
num=3.8
den=1.3
movie_mean=3.2
if available>=min_neighbors and den>0:
    used=available
    pred=num/den
    basis='cf'
else:
    used=0
    pred=movie_mean
    basis='movie-mean'
print(used,pred,basis)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q53=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q54 ·이웃 평균 제거 빈칸

각 이웃의 평점에서 그 이웃의 학습 평균을 빼도록 빈 칸을 채우고 최종 예측을 구하라.

```python
target_mean=3
sims=[0.25,0.75]
ratings=[5,2]
means=[4,3]
num=0
for j in range(2):
    deviation=ratings[j]-___
    num+=sims[j]*deviation
pred=target_mean+num/sum(sims)
print(pred)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q54=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q55 ·대상 평균 복원을 빠뜨린 오류

num은 이미 이웃 평균을 제거한 편차 가중합이다. 아래 코드를 평균보정 평점 예측으로 수정하고 원래 출력과 수정 출력의 의미를 설명하라.

```python
target_mean=3.5
num=0.4
den=0.8
pred=num/den
print(pred)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q55=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q56 ·마지막 범위 제한 빈칸

min(a,b)는 작은 값,max(a,b)는 큰 값을 반환한다. raw를 1 –5 범위로 제한하도록 두 빈칸을 채우라.

```python
raw=5.4
pred=min(___,max(___,raw))
print(raw,pred)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q56=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q57 ·아이템 CF의 평점 출처 오류

아이템 CF는 대상 사용자의 이웃 영화 평점을 사용한다. movies와 sims의 순서는 A,B다. 잘못 선택한 평점 딕셔너리를 수정하고 예측을 구하라.

```python
movies=['A','B']
sims=[0.75,0.25]
target_ratings={'A':2,'B':4}
other_ratings={'A':5,'B':1}
num=0
for j in range(2):
    num+=sims[j]*other_ratings[movies[j]]
pred=num/sum(sims)
print(pred)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q57=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q58 ·행과 열 추적

R의 행은 u1,u2,u3 이 고열은 A,B이다. None은 미관측이다. user _vector,item _vector의 값과 각각 어떤 대상을 나타내는지 쓰라.

```python
R=[[5,None],[3,4],[1,2]]
user_vector=R[1]
item_vector=[row[1] for row in R]
print(user_vector,item_vector)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q58=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q59 ·RMSE 마지막 제곱근 빈칸

같은 평가 관측의 실제 값과 예측 값이다. MSE에서 RMSE로 바꾸는 빈칸을 채우고 출력을 구하라.

```python
actual=[5,1,3]
pred=[4,3,3]
squared=[]
for j in range(len(actual)):
    squared.append((actual[j]-pred[j])**2)
mse=sum(squared)/len(actual)
rmse=___
print(round(rmse,4))
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q59=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


### Q60 ·평균 계산에 test가 들어간 오류

train은 학습 평점,test는 숨 긴 평가 정답이다. 평가 정답을 미리 보지 않도록 평균 계산을 수정하고 예측·MAE를 쓰라. 고정 평균 모델이다.

```python
train=[2,4]
test=[5,5]
mean=sum(train+test)/len(train+test)
mae=sum(abs(y-mean) for y in test)/len(test)
print(mean,mae)
```

기록: 바꿀 줄 또는 빈 칸 / 변수의 중간 값 / 최종 출력 / 연결되는 이론.


In [ ]:
my_answer_Q60=''  # 자신의 풀이를 기록하고 필요하면 별도 코드 셀을 추가하세요.


## 실행 결과 검증

시연의 실제 관측·예측·기여도·자료 분리를 검사합니다. 학생 연습 문제의 정답을 공개하는 검사가 아닙니다. 실패하면 오류를 확인하고 원인을 해결하세요.


In [ ]:
check_data=cf_review_data()
train_pairs=set(check_data['train'][['user_id','movie_id']].itertuples(index=False,name=None))
valid_pairs=set(check_data['validation'][['user_id','movie_id']].itertuples(index=False,name=None))
assert len(check_data['train'])==15 and len(check_data['validation'])==3
assert not train_pairs & valid_pairs
for axis,center in [('user',False),('user',True),('item',False)]:
    checked=cf_review_experiment(axis=axis,centered=center)
    terms=checked['contributions']
    base=checked['model'].user_means_.loc[1] if center else 0
    assert not terms.empty and math.isclose(terms.weight.sum(),1)
    assert math.isclose(terms.contribution.sum()+base,checked['details'].raw_prediction.iloc[0])
fallback=cf_review_experiment(min_neighbors=4)
assert fallback['contributions'].empty and fallback['details'].basis.iloc[0]=='movie-mean'
assert fallback['metrics']['rating_count']==3
import matplotlib.pyplot as plt
view=cf_review_view(report_fn=lambda r:'student callback reached')
assert 'student callback reached' in view[0]
plt.close(view[-1])
print('W07B-B RUNTIME CHECK PASSED')


## 점검 질문과 가져갈 생각

1. 미관측과 0점은 왜 다른가요?
2. 이웃을 고르기 전에 목표 영화 평가 여부를 왜 확인하나요?
3. 분자와 분모에 들어가는 이웃은 어떻게 맞추나요?
4. 평균보정은 누구의 평균을 빼고 누구의 평균을 더 하나요?
5. 설정 탐색 화면을 최종 test 성과라고 부르면 왜 문제인가요?

주교재 3장과 W03 –W04 기존 수업을 복습합니다. [코사인정의](https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity), [데이터누수](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage).


## 선택 확장 ·실제 MovieLens

핵심 실습은 합성 자료만으로 완료됩니다. 실제 데이터로 확장할 때는[기존W04B노트북](https://colab.research.google.com/github/lunalab-ai/recommender/blob/2026-fall-w04b/notebooks/student/w04b-cf-synthesis.ipynb)의 공식 다운로드·캐시·분할 절차를 사용하세요. 원자료 개인 속성이나 원평점 전체를 공개 출력하지 않고 요약 지표로 분석합니다.


## 마지막 ·직접 바꾸는 웹 애플리케이션

**LAB-B:** k =2 →1, 평균보정 off →on, 최소 이웃 1 →4, 축 user →item을 각각 비교하세요. 실행 전에 예상하고, 실행 후 이웃 표와 basis를 근거로 설명합니다. `my_report(result)`에 현재 k와 실제 CF 사용 이웃 수·근거를 읽어 표시하는 문장을 작성하세요. 힌트: `result["settings"]["k"]`, `result["details"].iloc[0].n_contributors`, `.basis`. 자가 점검: 설정 변경 후 숫자와 문장이 함께 변해야 합니다. 함수 수정 후 기존 앱을 닫고 다시 생성하세요.


In [ ]:
def my_report(result):
    return '설정을 바꾸기 전에 예상하고, 이웃표에서 근거를 찾아 설명하세요.'
app=build_cf_review_app(report_fn=my_report)


Colab에서는 아래 셀을 실행하면 내장 앱과 임시 공유 링크가 표시됩니다. 공유 링크는 만료될 수 있으므로 다음 복습에서는 고정 Colab 주소로 다시 시작합니다. 로컬 검증은 앱 생성·callback·브라우저 검사를 별도 기록합니다.


In [ ]:
if IN_COLAB:
    app.launch(share=True,debug=False,prevent_thread_lock=True)
else:
    print('Local runtime: app built; browser verification is recorded separately.')
